# CH03 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Nạp thư viện, dữ liệu Credit Default và bộ dữ liệu "đồ chơi" của Chương 2

In [ ]:
import sys
sys.path.append("code")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors
from common import credit_split, xgb_clf

pd.set_option("display.width", 80)
pd.set_option("display.precision", 3)
SEED = 42

X_train, X_test, y_train, y_test = credit_split()
X_toy, y_toy = make_classification(n_samples=600, n_features=2, n_redundant=0,
                                   n_clusters_per_class=1, weights=[0.9, 0.1],
                                   class_sep=1.0, random_state=3)
print("Credit -- train:", X_train.shape, "| Đồ chơi:", X_toy.shape, np.bincount(y_toy))


### Ô lệnh 2 -- "Hỏi" 10 láng giềng của một điểm thiểu số xem chúng thuộc lớp nào

In [ ]:
m = 10
nn_all = NearestNeighbors(n_neighbors=m + 1).fit(X_toy)    # tìm trong TOÀN BỘ dữ liệu
min_idx = np.where(y_toy == 1)[0]                          # vị trí 66 điểm thiểu số

for i in min_idx[[1, 32, 21]]:                             # xét 3 điểm làm ví dụ
    nb = nn_all.kneighbors(X_toy[[i]], return_distance=False)[0][1:]   # bỏ chính nó
    m_maj = (y_toy[nb] == 0).sum()                         # số láng giềng thuộc lớp đa số
    print(f"Điểm {i:3d}: nhãn láng giềng = {y_toy[nb]} -> {m_maj}/10 thuộc lớp đa số")


### Ô lệnh 3 -- Phân loại mọi điểm thiểu số thành SAFE / DANGER / NOISE

In [ ]:
def minority_types(X, y, m=10):
    """Trả về loại của từng điểm thiểu số theo quy tắc của Han và cộng sự (2005)."""
    nn = NearestNeighbors(n_neighbors=m + 1).fit(X)
    nb = nn.kneighbors(X[y == 1], return_distance=False)[:, 1:]
    m_maj = (y[nb] == 0).sum(axis=1)
    types = np.where(m_maj == m, "NOISE",
                     np.where(m_maj >= m / 2, "DANGER", "SAFE"))
    return types, m_maj

types_toy, mmaj_toy = minority_types(X_toy, y_toy)
print(pd.Series(types_toy).value_counts().to_dict())


### Ô lệnh 4 -- Vẽ ba loại điểm thiểu số lên mặt phẳng

In [ ]:
X_min = X_toy[y_toy == 1]
style = {"SAFE": ("#1baf7a", "o"), "DANGER": ("#eb6834", "D"), "NOISE": ("#e34948", "s")}

fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.scatter(*X_toy[y_toy == 0].T, s=8, color="#c9c8c2", label="Lớp đa số")
for t, (c, mk) in style.items():
    ax.scatter(*X_min[types_toy == t].T, s=22, color=c, marker=mk, edgecolor="white",
               linewidth=0.4, label=f"{t} ({(types_toy == t).sum()})")
ax.legend(fontsize=7, loc="lower right")
plt.savefig("figures/ch03_s04_types.pdf", bbox_inches="tight")


### Ô lệnh 5 -- Tự viết Borderline-SMOTE-1: chỉ sinh mẫu từ các điểm DANGER

In [ ]:
def borderline_smote_tay(X_min, types, G, k=5, seed=SEED):
    rng = np.random.default_rng(seed)
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    danger = np.where(types == "DANGER")[0]          # chỉ những điểm này được làm "hạt giống"
    new = []
    for _ in range(G):
        i = rng.choice(danger)                       # KHÁC SMOTE: chỉ chọn trong DANGER
        j = rng.choice(idx[i][1:])                   # láng giềng vẫn là mọi điểm thiểu số
        lam = rng.uniform(0, 1)
        new.append(X_min[i] + lam * (X_min[j] - X_min[i]))
    return np.array(new)

G = (y_toy == 0).sum() - (y_toy == 1).sum()
X_syn_b1 = borderline_smote_tay(X_min, types_toy, G)
print("Số điểm DANGER làm hạt giống:", (types_toy == "DANGER").sum(), "| số điểm sinh ra:", len(X_syn_b1))


### Ô lệnh 6 -- So sánh SMOTE, Borderline-1 và Borderline-2 (thư viện imblearn)

In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE

samplers = {"SMOTE": SMOTE(random_state=SEED),
            "Borderline-1": BorderlineSMOTE(kind="borderline-1", random_state=SEED),
            "Borderline-2": BorderlineSMOTE(kind="borderline-2", random_state=SEED)}
synth = {}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True, sharey=True)
for ax, (name, s) in zip(axes, samplers.items()):
    Xr, yr = s.fit_resample(X_toy, y_toy)
    synth[name] = Xr[len(X_toy):]                     # các điểm mới nằm ở cuối mảng
    ax.scatter(*X_toy[y_toy == 0].T, s=6, color="#c9c8c2")
    ax.scatter(*X_min.T, s=12, color="#eb6834")
    ax.scatter(*synth[name].T, s=8, marker="x", color="#2a78d6", linewidth=0.7)
    ax.set_title(f"{name} ({len(synth[name])} điểm mới)")
plt.savefig("figures/ch03_s06_compare.pdf", bbox_inches="tight")


### Ô lệnh 7 -- Đo xem điểm tổng hợp "lấn" vào vùng của lớp đa số đến mức nào

In [ ]:
nn10 = NearestNeighbors(n_neighbors=10).fit(X_toy)       # chỉ dùng dữ liệu GỐC
for name, S_ in synth.items():
    nb = nn10.kneighbors(S_, return_distance=False)
    share_maj = (y_toy[nb] == 0).mean(axis=1)              # tỷ lệ láng giềng thuộc lớp đa số
    print(f"{name:13s}: TB {share_maj.mean():.0%} láng giềng là lớp đa số;"
          f" {np.mean(share_maj > 0.5):.0%} điểm nằm hẳn trong vùng đa số")


### Ô lệnh 8 -- Chẩn đoán dữ liệu thật: khách hàng vỡ nợ thuộc loại nào?

In [ ]:
from sklearn.preprocessing import StandardScaler

Z_train = StandardScaler().fit_transform(X_train)          # chuẩn hóa trước khi tính khoảng cách
types_c, mmaj_c = minority_types(Z_train, y_train.values)
ratio = pd.Series(types_c).value_counts(normalize=True)
print("Credit Default:", (ratio * 100).round(1).to_dict(), "(%)")
toy_ratio = pd.Series(types_toy).value_counts(normalize=True)
print("Dữ liệu đồ chơi:", (toy_ratio * 100).round(1).to_dict(), "(%)")


### Ô lệnh 9 -- Phân bố số láng giềng thuộc lớp đa số (m') của mọi khách hàng vỡ nợ

In [ ]:
counts = pd.Series(mmaj_c).value_counts().sort_index()
print(counts.to_dict())

colors = ["#1baf7a"] * 5 + ["#eb6834"] * 5 + ["#e34948"]   # SAFE: 0-4, DANGER: 5-9, NOISE: 10
fig, ax = plt.subplots(figsize=(5.5, 2.6))
ax.bar(counts.index, counts.values, color=[colors[i] for i in counts.index])
ax.set_xticks(range(11))
ax.set_xlabel("m' = số láng giềng (trong 10) thuộc lớp KHÔNG vỡ nợ")
ax.set_ylabel("Số khách hàng vỡ nợ")
plt.savefig("figures/ch03_s09_mmaj.pdf", bbox_inches="tight")


### Ô lệnh 10 -- So sánh 4 chiến lược lấy mẫu với 2 mô hình (đánh giá chéo, ~1 phút)

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"PR-AUC": "average_precision", "F1": "f1", "Recall": "recall", "Precision": "precision"}
strategies = {"Không lấy mẫu": None, **samplers}

def make(model, sampler):
    steps = [("scale", StandardScaler())]
    if sampler is not None:
        steps.append(("sampler", sampler))
    return Pipeline(steps + [("model", model)])

cv_rows = {}
for mname, mk in {"LogReg": lambda: LogisticRegression(max_iter=2000), "XGBoost": xgb_clf}.items():
    for sname, s in strategies.items():
        r = cross_validate(make(mk(), s), X_train, y_train, cv=skf, scoring=scoring)
        cv_rows[f"{mname} | {sname}"] = {k: r["test_" + k].mean() for k in scoring}
cv_df = pd.DataFrame(cv_rows).T
print(cv_df)


### Ô lệnh 11 -- Biểu đồ cột cho XGBoost: xu hướng khi càng "tập trung vào biên"

In [ ]:
xg = cv_df[cv_df.index.str.startswith("XGBoost")]
xg.index = [i.split("| ")[1] for i in xg.index]

ax = xg.T.plot(kind="bar", figsize=(7, 2.8), width=0.8, rot=0,
               color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])
ax.set_ylim(0.3, 0.72); ax.set_ylabel("Điểm CV trung bình")
ax.legend(ncol=4, fontsize=7, loc="upper center")
plt.savefig("figures/ch03_s11_bars.pdf", bbox_inches="tight")


### Ô lệnh 12 -- Kiểm tra trên tập test (XGBoost, ngưỡng 0,5)

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

test_rows, proba = {}, {}
for sname, s in strategies.items():
    pipe = make(xgb_clf(), s).fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    proba[sname] = p
    pred = (p >= 0.5).astype(int)
    test_rows[sname] = {"PR-AUC": average_precision_score(y_test, p), "F1": f1_score(y_test, pred),
                        "Recall": recall_score(y_test, pred), "Precision": precision_score(y_test, pred)}
print(pd.DataFrame(test_rows).T)


### Ô lệnh 13 -- Đối chiếu với phương án dời ngưỡng của Chương 2 (t* = 0,26)

In [ ]:
t_star = 0.26                                   # ngưỡng tìm được bằng out-of-fold ở Chương 2
pred = (proba["Không lấy mẫu"] >= t_star).astype(int)
print(f"XGBoost, t = 0,26 -> F1 = {f1_score(y_test, pred):.3f}, "
      f"Recall = {recall_score(y_test, pred):.3f}, Precision = {precision_score(y_test, pred):.3f}")


### Ô lệnh 14 -- Khi nào Borderline-SMOTE thắng SMOTE? Thử với các mức chồng lấn khác nhau

In [ ]:
rows = []
for sep in [0.5, 1.0, 1.5, 2.0]:                 # class_sep càng lớn, hai lớp càng tách biệt
    Xs, ys = make_classification(n_samples=4000, n_features=10, n_informative=5,
                                 weights=[0.9, 0.1], class_sep=sep, flip_y=0, random_state=0)
    t, _ = minority_types(StandardScaler().fit_transform(Xs), ys)
    row = {"class_sep": sep, "% DANGER": np.mean(t == "DANGER") * 100}
    for sname in ["SMOTE", "Borderline-1"]:
        r = cross_validate(make(xgb_clf(), samplers[sname]), Xs, ys, cv=skf, scoring=scoring)
        row[f"F1 {sname}"] = r["test_F1"].mean()
        row[f"PR-AUC {sname}"] = r["test_PR-AUC"].mean()
    rows.append(row)
print(pd.DataFrame(rows).set_index("class_sep").round(3).to_string())
